In [ ]:
###
# セットアップ
###

from pathlib import Path
import random
import shutil
import sys
import urllib.request
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as F

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT_PATH = Path("/content/drive/MyDrive/cnn-hands-on")
except Exception:
    ROOT_PATH = Path.cwd()

if str(ROOT_PATH) not in sys.path:
    sys.path.append(str(ROOT_PATH))

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("ROOT_PATH:", ROOT_PATH)
print("device:", device)

random.seed(42)
np.random.seed(42)
torch.manual_seed(42)


In [ ]:
###
# Penn-Fudan Pedestrian データセットをダウンロードする
###

DATA_DIR = ROOT_PATH / "data" / "PennFudanPed"
ZIP_PATH = ROOT_PATH / "data" / "PennFudanPed.zip"
DATA_URL = "https://www.cis.upenn.edu/~jshi/ped_html/PennFudanPed.zip"

if not DATA_DIR.exists():
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    if not ZIP_PATH.exists():
        print("downloading dataset...")
        urllib.request.urlretrieve(DATA_URL, ZIP_PATH)

    print("extracting dataset...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(DATA_DIR.parent)

    extracted_dir = DATA_DIR.parent / "PennFudanPed"
    if extracted_dir.exists() and extracted_dir != DATA_DIR:
        if DATA_DIR.exists():
            shutil.rmtree(DATA_DIR)
        extracted_dir.rename(DATA_DIR)

print("DATA_DIR:", DATA_DIR)
print("images:", len(list((DATA_DIR / "PNGImages").glob("*.png"))))
print("masks:", len(list((DATA_DIR / "PedMasks").glob("*.png"))))


In [ ]:
###
# データセットを読み込む
###

class PennFudanSegmentationDataset(Dataset):
    def __init__(self, root_dir, split="train", split_seed=42):
        self.root_dir = Path(root_dir)
        self.image_paths = sorted((self.root_dir / "PNGImages").glob("*.png"))
        self.mask_paths = sorted((self.root_dir / "PedMasks").glob("*.png"))

        assert len(self.image_paths) == len(self.mask_paths)

        indices = np.arange(len(self.image_paths))
        rng = np.random.default_rng(split_seed)
        rng.shuffle(indices)

        n_total = len(indices)
        n_train = int(n_total * 0.70)
        n_valid = int(n_total * 0.15)

        split_map = {
            "train": indices[:n_train],
            "valid": indices[n_train:n_train + n_valid],
            "test": indices[n_train + n_valid:],
        }

        if split not in split_map:
            raise ValueError(f"unknown split: {split}")

        self.indices = list(split_map[split])

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        image = Image.open(self.image_paths[real_idx]).convert("RGB")
        mask_array = np.array(Image.open(self.mask_paths[real_idx]))

        if mask_array.ndim == 3:
            mask_array = mask_array.max(axis=2)

        # 0: background, 1: pedestrian
        mask = (mask_array > 0).astype(np.int64)

        image_tensor = F.to_tensor(image)
        mask_tensor = torch.from_numpy(mask)
        return image_tensor, mask_tensor


train_dataset = PennFudanSegmentationDataset(DATA_DIR, split="train")
valid_dataset = PennFudanSegmentationDataset(DATA_DIR, split="valid")
test_dataset = PennFudanSegmentationDataset(DATA_DIR, split="test")

# 画像サイズが異なるため、9回目と同じく collate_fn でまとめる
def collate_fn(batch):
    return tuple(zip(*batch))

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True, num_workers=2, collate_fn=collate_fn)
valid_loader = DataLoader(valid_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)
test_loader = DataLoader(test_dataset, batch_size=4, shuffle=False, num_workers=2, collate_fn=collate_fn)

print("train:", len(train_dataset))
print("valid:", len(valid_dataset))
print("test:", len(test_dataset))


In [ ]:
###
# サンプルを確認する
###

CLASS_NAMES = {0: "background", 1: "person"}


def show_sample(image_tensor, mask_tensor):
    fig, axes = plt.subplots(1, 2, figsize=(8, 4))
    axes[0].imshow(image_tensor.permute(1, 2, 0))
    axes[0].set_title("image")
    axes[0].axis("off")

    axes[1].imshow(mask_tensor, vmin=0, vmax=1, cmap="gray")
    axes[1].set_title("mask")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


sample_image, sample_mask = train_dataset[0]
show_sample(sample_image, sample_mask)


In [ ]:
###
# 軽量なセグメンテーションモデルを定義する
###

class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class TinyUNet(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        self.enc1 = ConvBlock(3, 16)
        self.pool1 = nn.MaxPool2d(2)
        self.enc2 = ConvBlock(16, 32)
        self.pool2 = nn.MaxPool2d(2)
        self.bottleneck = ConvBlock(32, 64)
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)
        self.dec2 = ConvBlock(64, 32)
        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=2, stride=2)
        self.dec1 = ConvBlock(32, 16)
        self.head = nn.Conv2d(16, num_classes, kernel_size=1)

    def forward(self, x):
        skip1 = self.enc1(x)
        skip2 = self.enc2(self.pool1(skip1))
        x = self.bottleneck(self.pool2(skip2))
        x = self.up2(x)
        if x.shape[-2:] != skip2.shape[-2:]:
            x = torch.nn.functional.interpolate(x, size=skip2.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip2], dim=1)
        x = self.dec2(x)
        x = self.up1(x)
        if x.shape[-2:] != skip1.shape[-2:]:
            x = torch.nn.functional.interpolate(x, size=skip1.shape[-2:], mode="bilinear", align_corners=False)
        x = torch.cat([x, skip1], dim=1)
        x = self.dec1(x)
        return self.head(x)


model = TinyUNet(num_classes=2).to(device)
print(model)


In [ ]:
###
# 学習する
###

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
num_epochs = 8
best_val_loss = float("inf")
MODEL_DIR = ROOT_PATH / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
save_path = MODEL_DIR / "10_segmentation_tiny_unet.pth"


def evaluate(loader):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_pixels = 0
    iou_scores = []

    with torch.no_grad():
        for batch_images, batch_masks in loader:
            for image, mask in zip(batch_images, batch_masks):
                image = image.to(device).unsqueeze(0)
                mask = mask.to(device).unsqueeze(0)
                logits = model(image)
                loss = criterion(logits, mask)
                total_loss += loss.item()

                preds = logits.argmax(dim=1)
                total_correct += (preds == mask).sum().item()
                total_pixels += mask.numel()

                pred_fg = preds == 1
                true_fg = mask == 1
                intersection = (pred_fg & true_fg).sum().item()
                union = (pred_fg | true_fg).sum().item()
                if union > 0:
                    iou_scores.append(intersection / union)

    mean_loss = total_loss / len(loader.dataset)
    pixel_acc = total_correct / total_pixels
    mean_iou = float(np.mean(iou_scores)) if iou_scores else 0.0
    return mean_loss, pixel_acc, mean_iou


for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for batch_images, batch_masks in train_loader:
        for image, mask in zip(batch_images, batch_masks):
            image = image.to(device).unsqueeze(0)
            mask = mask.to(device).unsqueeze(0)

            optimizer.zero_grad()
            logits = model(image)
            loss = criterion(logits, mask)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

    train_loss = running_loss / len(train_loader.dataset)
    val_loss, val_acc, val_iou = evaluate(valid_loader)

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), save_path)

    print(
        f"Epoch {epoch+1:02d}/{num_epochs} | "
        f"train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | "
        f"val_acc={val_acc:.4f} | val_iou={val_iou:.4f}"
    )

print("best val loss:", best_val_loss)
print("saved to:", save_path)


In [ ]:
###
# 推論と可視化
###

model.load_state_dict(torch.load(save_path, map_location=device))
model.eval()


def overlay_mask(image_tensor, mask_tensor, alpha=0.45):
    image = (image_tensor.permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    overlay = image.copy()
    colors = {1: np.array([65, 105, 225], dtype=np.uint8)}
    for class_id, color in colors.items():
        pixels = mask_tensor.cpu().numpy() == class_id
        overlay[pixels] = ((1 - alpha) * overlay[pixels] + alpha * color).astype(np.uint8)
    return overlay


fig, axes = plt.subplots(3, 3, figsize=(12, 12))
for row_idx, idx in enumerate([0, 1, 2]):
    image, mask = test_dataset[idx]
    with torch.no_grad():
        pred = model(image.unsqueeze(0).to(device)).argmax(dim=1)[0].cpu()

    axes[row_idx, 0].imshow(image.permute(1, 2, 0))
    axes[row_idx, 0].set_title("image")
    axes[row_idx, 0].axis("off")

    axes[row_idx, 1].imshow(overlay_mask(image, mask))
    axes[row_idx, 1].set_title("ground truth")
    axes[row_idx, 1].axis("off")

    axes[row_idx, 2].imshow(overlay_mask(image, pred))
    axes[row_idx, 2].set_title("prediction")
    axes[row_idx, 2].axis("off")

plt.tight_layout()
plt.show()

test_loss, test_acc, test_iou = evaluate(test_loader)
print(f"test_loss={test_loss:.4f}")
print(f"test_acc={test_acc:.4f}")
print(f"test_iou={test_iou:.4f}")
